In [ ]:
from pathlib import Path
import json
import sys

from neuvueclient import NeuvueQueue
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

sys.path.append(str(Path.cwd().parents[1]))
from utils.constants import NEUVUE_QUEUE_URL, SPELUNKER_URL, NEUVUE_NG_URL

In [ ]:
client = NeuvueQueue(NEUVUE_QUEUE_URL)

## Generic Forced Choice Task

Task type for a forced choice task.

In [ ]:
# Change these as needed
AUTHOR = "dxenes1"
NAMESPACE = "forcedChoiceExample"
ASSIGNEES = ["unassigned_novice"]
INSTRUCTIONS = {"prompt": "Is the selected seg ID a multi-soma? (two valid neurons merged together)"}


### Grab the necessary input data, in this case we are just using the nucleus detection table.

In [ ]:
from caveclient import CAVEclient
cave = CAVEclient('minnie65_phase3_v1')

In [ ]:
nucleus_neuron_svm = cave.materialize.query_table('nucleus_neuron_svm')
nucleus_neuron_svm['pt_position_8nm'] = nucleus_neuron_svm['pt_position'].apply(lambda x: np.array(x) // np.array([2,2,1])) 

In [ ]:
# Filter only neurons and get multi-soma neurons by using value counts on the root IDs
neuron_nuclei = nucleus_neuron_svm[nucleus_neuron_svm['cell_type'] == 'neuron']
multi_soma_neurons = neuron_nuclei['pt_root_id'].value_counts()[neuron_nuclei['pt_root_id'].value_counts() >= 2]

# Remove seg ID 0 and 864691135763133238 (the MOAME - mother of all merge errors)
multi_soma_neurons.drop([0, 864691135763133238], inplace=True)



### Create task dataframe to more easily create and assign tasks.

In [ ]:
task_rows = []
for root_id in multi_soma_neurons.index.unique():
    nuclei_subset = neuron_nuclei[neuron_nuclei['pt_root_id']==root_id]
    row = {}
    row['pt_root_id'] = root_id
    row['proofreaders'] = ASSIGNEES
    row['metadata'] = {'nuclei': nuclei_subset['id'].tolist()}
    row['pt_position'] = nuclei_subset.iloc[0].pt_position_8nm
    row['annos'] = nuclei_subset['pt_position_8nm'].tolist()
    row['instructions'] = INSTRUCTIONS
    task_rows.append(row)
    
task_df = pd.DataFrame(task_rows)

### We use a function in ng_utils.py to parse this dataframe and create custom neuroglancer states for each task.

In [ ]:
from utils.ng_state_utils import add_basic_ng_states_with_anno_layer_and_points_to_df, set_ng_url

# For Spelunker encoded state
set_ng_url(SPELUNKER_URL)

# For Neuvue Neuroglancer encoded state (Neuvue Legacy in the Neuvue Admin Console)
# set_ng_url(NEUVUE_NG_URL)

add_basic_ng_states_with_anno_layer_and_points_to_df(task_df, client=cave) # Passing in CAVE is necessary for Spelunker-encoded annotation layer

In [ ]:
task_df["states"][0]

### We finally upload the tasks using another convenience function which reads in the data frame and creates the tasks sequentially.

The NeuVue client will automatically upload JSON state strings to the JSON state server as its creating the tasks. 

In [ ]:
from utils.post_tasks_utils import post_tasks_from_state_and_proofreader_df
post_tasks_from_state_and_proofreader_df(
    neuvue_client=client, 
    state_df=task_df, 
    author=AUTHOR, 
    namespace=NAMESPACE
)
print("Done")

Check your work

## Delete all tasks
If something went wrong you can use this snippet to delete all tasks from this namespace

In [ ]:
# tasks = client.get_tasks(sieve={'namespace': NAMESPACE, 'assignee': ASSIGNEES}, select=['_id'])
# for _id in tasks.index:
#     client.delete_task(_id)